# Imports

In [67]:
"""QUICK PRELIMINARY STEP"""
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if "microbiome2function" in repo_root.parts:
    necessary_depth = repo_root.parts.index("microbiome2function")
    M2F_loc = Path(*repo_root.parts[:necessary_depth+1])
else:
    # NOTE: fallback when the kernel starts outside the repo
    M2F_loc = next((p for p in repo_root.parents if p.name == "microbiome2function"), repo_root)

if str(M2F_loc) not in sys.path:
    sys.path.insert(0, str(M2F_loc))  # keep explicit for local imports


In [ ]:
from torch_geometric.data import InMemoryDataset, Data
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import MultiLabelBinarizer
import torch
import pandas as pd
import numpy as np
import gdown

from pathlib import Path
import zipfile
import re
import os

from M2F import fetch_uniprotkb_fields, util, get_GODag

# Globals

In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQUENCE_ENCODER_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
ID_TO_ACCESSION_FILE_NAME = "uniref_index_count.csv" # uniref,i
FEATURES = ["accession", "sequence", "go_f"]

#### NOTE: [available uniprot features](https://www.uniprot.org/help/return_fields)

# Feature encoders

## Sequence

In [ ]:
class SequenceEncoder:
    def __init__(self, model_name: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()

    def __call__(self, sequences: list[str] | str) -> torch.Tensor:
        if isinstance(sequences, str):
            # allow single-sequence inputs (e.g., row-wise transforms)
            sequences = [sequences]
        toks = self.tokenizer(
            sequences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
            return_special_tokens_mask=True,
        )
        special_mask = toks.pop("special_tokens_mask")
        toks = {k: v.to(device) for k, v in toks.items()}
        special_mask = special_mask.to(device)

        with torch.no_grad():
            outputs = self.model(**toks)
            embeddings = outputs.last_hidden_state    # [B, L, D]
            attn_mask = toks["attention_mask"].bool() # [B, L]
            spec_mask = special_mask.bool()           # [B, L]
            keep_mask = attn_mask & (~spec_mask)      # [B, L]
            #                           V -- add an empty axis at the end
            keep_mask_f = keep_mask.unsqueeze(-1).type_as(embeddings) # [B, L, 1]
            #                    V -- each entry in L is broadcasted into D dims
            summed = (embeddings * keep_mask_f).sum(dim=1) # [B, L, D] * [B, L, 1] = ([B, L, D] * [B, L, D]).sum(dim=1) = [B, D]

            # Given [B, L], sum out the masks for each sentence of length L (1st axis):
            # [B, L] -> [B,], where each entry in B is a count of real tokens inside the sequence corresponding to its position in the batch array;
            # also clip the lower bound of the output sum at 1 so that if there are no real tokens, we don't end up with a 0
            counts = keep_mask.sum(dim=1).clamp(min=1).unsqueeze(-1) # Now, add an empty axis at the end [B,] -> [B, 1]
            pooled = summed / counts # [B, D] / [B, 1] = [B, D] / [B, D] (broadcast each out of the B divisors into D dims, so that we can divide each entry of the summed embedding)
        return pooled


# Data interface

## Transforms and Filters

In [ ]:
def pre_transform_factory(model_name: str, required_GO_depth: int):
    
    SE = SequenceEncoder(model_name)
    godag = get_GODag()

    def collapse_to_depth(go_ids: tuple[str, ...], k: int, *, godag) -> tuple[str]:
        kept = set()
        for gid in go_ids:
            if gid not in godag:
                continue
            node = godag[gid]
            ancestors = {gid}.union(node.get_all_parents())
            at_k = {n for n in ancestors if godag[n].depth == k}
            kept.update(at_k if at_k else {min(ancestors, key=lambda x: godag[x].depth)})
        return tuple(sorted(kept)) if kept else np.nan

    def inner(row: pd.Series):
        nonlocal SE
        nonlocal required_GO_depth
        nonlocal godag

        row["Sequence"] = SE(row["Sequence"])
        row["Gene Ontology (molecular function)"] = \
            collapse_to_depth(row["Gene Ontology (molecular function)"],
                              required_GO_depth,
                              godag=godag)

    return inner

def pre_filter(row: pd.Series):
    return not np.isnan(row["Gene Ontology (molecular function)"])

def transform_factory(go2cls_map: dict):

    def inner(row: pd.Series):
        nonlocal go2cls_map
        go_ids = row["Gene Ontology (molecular function)"]
        if isinstance(go_ids, float) and np.isnan(go_ids):
            # NOTE: keep row unchanged if targets are missing
            return row
        if isinstance(go_ids, str):
            # NOTE: allow single GO id to flow through
            go_ids = (go_ids,)

        dim = len(go2cls_map)
        idx = [go2cls_map[g] for g in go_ids if g in go2cls_map]
        target = torch.zeros((dim,), dtype=torch.float32)
        if idx:
            labels = torch.tensor(sorted(set(idx)), dtype=torch.long)
            target.scatter_(0, labels, 1.0)

        row["Gene Ontology (molecular function)"] = target
        return row

    return inner


## RAM-backed

In [ ]:
class Proteins_KG_InMem(InMemoryDataset):

    _UNIREF90_RE   = re.compile(r"UniRef90_([A-Z0-9]+)")
    _URL = "https://drive.google.com/file/d/1YU1G4xCcJHdXixZs7Ai801Xx4ckwZ4hz"

    def __init__(self,
                root: str,
                features: list[str]|None=None,
                transform=None,
                pre_transform=None,
                pre_filter=None,
                uniprot_query_batch_size=8):
        super().__init__(root, transform, pre_transform, pre_filter)
        self.features = list(set(features or []) | {"accession"})
        self.uniprot_query_batch_size = uniprot_query_batch_size

        # None before `self.download` was called
        self.node_id2accession: dict[int, str] | None = None
        self.go2cls_id: dict[str, int] | None = None

        if os.path.exists(self.processed_paths[0]):
            self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_dir(self):
        return self.root / Path("raw")

    @staticmethod
    def _clean_up(*files: Path):
        for f in files:
            if f and Path(f).exists():
                os.remove(f)

    @property
    def raw_file_names(self):
        # Minimal sentinel; chunk_*.csv are expected to be present as well
        return [ID_TO_ACCESSION_FILE_NAME]

    def download(self):
        out_dir = Path(self.raw_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        index_path = out_dir / ID_TO_ACCESSION_FILE_NAME
        if not index_path.exists():
            # download the compressed edge_attrs and nodes
            name = gdown.download(Proteins_KG_InMem._URL, str(out_dir), quiet=False, fuzzy=True)

            # uncompress into the root/raw directory
            zip_path = Path(name)
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(path=out_dir)

            # clean up only the downloaded archive
            self._clean_up(zip_path)

        # extract accession IDs
        accessions = dict()
        for row in pd.read_csv(index_path).itertuples(index=False):
            accession = Proteins_KG_InMem._UNIREF90_RE.match(row.uniref).group(1)
            if not accession.startswith(("UNK", "UPI")):
                accessions[int(row.i)] = accession
        self.node_id2accession = accessions

        node_features_path = out_dir / "node_features.csv"
        if not node_features_path.exists():
            fetched = fetch_uniprotkb_fields(list(accessions.values()),
                                             self.features,
                                             request_size=self.uniprot_query_batch_size,
                                             rps=1,
                                             max_retry=20)
            fetched.to_csv(node_features_path, index=False)

    def process(self):
        out_dir = Path(self.raw_dir)
        node_features_path = out_dir / "node_features.csv"
        index_path = out_dir / ID_TO_ACCESSION_FILE_NAME

        # features and targets
        if node_features_path.exists():
            XY = pd.read_csv(node_features_path)
        else:
            raise NameError("node_features.csv is not present in raw_file_names")

        # NOTE: normalize column names to match downstream transforms
        col_map = {
            "sequence": "Sequence",
            "go_f": "Gene Ontology (molecular function)",
        }
        XY = XY.rename(columns={k: v for k, v in col_map.items() if k in XY.columns})

        # align to the UniRef index if needed
        index_df = pd.read_csv(index_path)
        node_ids = index_df["i"].to_numpy(dtype=np.int64)  # 1-based
        if len(XY) != len(node_ids):
            if "accession" in XY.columns:
                index_df = index_df.assign(accession=index_df["uniref"].str.replace("UniRef90_", "", regex=False))
                XY = index_df.merge(XY, on="accession", how="left", sort=False)
                node_ids = XY["i"].to_numpy(dtype=np.int64)
            else:
                raise ValueError("node_features.csv row count does not match index file and no accession column to align")

        XY["_node_id"] = node_ids - 1  # convert to 0-based ids

        if self.pre_transform:
            def _apply(row: pd.Series):
                out = self.pre_transform(row)
                return row if out is None else out
            XY = XY.apply(_apply, axis=1)

        kept_node_ids = XY["_node_id"].to_numpy(dtype=np.int64)
        if self.pre_filter:
            mask = XY.apply(self.pre_filter, axis=1)
            XY = XY[mask].copy()
            kept_node_ids = XY["_node_id"].to_numpy(dtype=np.int64)

        # build a GO -> class id map (flattening tuples)
        go_set = set()
        for v in XY["Gene Ontology (molecular function)"]:
            if isinstance(v, (list, tuple, set)):
                go_set.update(v)
            elif pd.notna(v):
                go_set.add(v)
        self.go2cls_id = {go: i for i, go in enumerate(sorted(go_set))}
        row_transform = transform_factory(self.go2cls_id)
        XY = XY.apply(row_transform, axis=1)

        X = XY["Sequence"]
        Y = XY["Gene Ontology (molecular function)"]

        # edges and their attrs
        p = re.compile(r"chunk_(\d+)\.csv")
        edge_src = []
        edge_dst = []
        edge_w = []
        for file in util.files_from(out_dir, r"chunk_\d+\.csv"):
            match = p.match(Path(file).name)
            if not match:
                continue
            i = int(match.group(1)) - 1  # NOTE: convert 1-based ids to 0-based
            row = pd.read_csv(file)
            if row.empty:
                continue
            idx = row["j"].to_numpy(dtype=np.int64) - 1
            weights = row["v"].to_numpy(dtype=np.float32)
            degree = len(idx)
            edge_src.append(np.full(degree, i, dtype=np.int64))
            edge_dst.append(idx)
            edge_w.append(weights)

        if edge_src:
            edge_index = np.vstack([np.concatenate(edge_src), np.concatenate(edge_dst)])
            edge_attr = np.concatenate(edge_w).reshape(-1, 1)
        else:
            edge_index = np.empty((2, 0), dtype=np.int64)
            edge_attr = np.empty((0, 1), dtype=np.float32)

        # If nodes were filtered out, drop and reindex edges to keep consistency
        if self.pre_filter:
            max_id = int(node_ids.max()) - 1
            id_map = -np.ones(max_id + 1, dtype=np.int64)
            id_map[kept_node_ids] = np.arange(len(kept_node_ids))
            if edge_index.size:
                keep_mask = (id_map[edge_index[0]] >= 0) & (id_map[edge_index[1]] >= 0)
                edge_index = id_map[edge_index[:, keep_mask]]
                edge_attr = edge_attr[keep_mask]

        # stack node features and targets
        x_vals = [t.squeeze(0) if hasattr(t, "ndim") and t.ndim == 2 and t.shape[0] == 1 else t for t in X]
        x = torch.vstack(x_vals)
        y = torch.vstack([t for t in Y])

        # ensure we can map back to accessions (0-based ids)
        if self.node_id2accession is None:
            accessions = dict()
            for row in index_df.itertuples(index=False):
                accession = Proteins_KG_InMem._UNIREF90_RE.match(row.uniref).group(1)
                if not accession.startswith(("UNK", "UPI")):
                    accessions[int(row.i) - 1] = accession
            self.node_id2accession = accessions
        else:
            self.node_id2accession = {int(k) - 1: v for k, v in self.node_id2accession.items()}

        if self.pre_filter:
            self.node_id2accession = {
                int(id_map[old_id]): acc
                for old_id, acc in self.node_id2accession.items()
                if id_map[old_id] >= 0
            }

        KG = Data(
            x=x,
            edge_index=torch.from_numpy(edge_index).long(),
            edge_attr=torch.from_numpy(edge_attr).float(),
            y=y
        )
        KG.go2cls_id = self.go2cls_id
        KG.node_id2accession = self.node_id2accession

        data, slices = self.collate([KG])
        torch.save((data, slices), self.processed_paths[0])

    @property
    def processed_file_names(self):
        return "KG.pt"

## DB-backed

In [72]:
...

Ellipsis

# Graph Encoder

In [73]:
...

Ellipsis

# Main model

In [74]:
...

Ellipsis

# Training Loop

## Preliminaries

### Transforms, dataset, data loader, model, optimizer, criterion, learning rate scheduler

In [75]:
...

Ellipsis

### Evaluation metrics

In [76]:
...

Ellipsis

## Loop

In [77]:
...

Ellipsis

# Final report on the model prototype number 1

...